# 02 - Analiza w czasie rzeczywistym (Kafka)

Podgląd surowego strumienia telemetrii bezpośrednio z Kafki z poziomu notebooka, bez uruchamiania pełnego joba Spark Streaming. Przydatne do szybkiego debugowania danych wejściowych.

In [ ]:
import json
import os
from collections import deque

from kafka import KafkaConsumer

BOOTSTRAP_SERVERS = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092")
TOPIC = "f1-telemetry"

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP_SERVERS,
    auto_offset_reset="latest",
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=15000,
)

In [ ]:
buffer = deque(maxlen=500)

for message in consumer:
    buffer.append(message.value)
    print(message.value)

## Szybkie agregacje w pamięci (pandas)

Po zebraniu kilkunastu sekund wiadomości w `buffer` można policzyć bieżące statystyki per kierowca, analogicznie do agregacji z `spark_jobs/streaming_job.py`.

In [ ]:
import pandas as pd

live_df = pd.DataFrame(list(buffer))
live_df.groupby("driver_id").agg(
    avg_speed=("speed_kmh", "mean"),
    max_speed=("speed_kmh", "max"),
    max_tire_wear=("tire_wear_pct", "max"),
)

## Pełna analiza strumieniowa w Spark

Do ciągłej, skalowalnej analizy (okna czasowe, watermarking, zapis do Parquet) użyj joba `spark_jobs/streaming_job.py` uruchamianego przez `spark-submit` na klastrze Spark (patrz README).